In [1]:
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import models, transforms
from torchvision.models import ResNet50_Weights, DenseNet121_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import models, transforms
from torchvision.models import ResNet50_Weights, DenseNet121_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [4]:
DATA_ROOT = Path("./two_class_cifar")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT.resolve())
print("Using device:", device)

Data root: /home/amyliu/Desktop/GR/two_class_cifar
Using device: cuda


In [5]:
image_paths = []
labels = []

label_to_idx = {"cat": 0, "dog": 1}

for label_name in ["cat", "dog"]:
    folder = DATA_ROOT / label_name
    for img_path in sorted(folder.glob("*.png")):
        image_paths.append(str(img_path))
        labels.append(label_to_idx[label_name])

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Total images:", len(image_paths))
print("Label counts:", pd.Series(labels).value_counts().sort_index().to_dict())

Total images: 40
Label counts: {0: 20, 1: 20}


In [6]:
class SimpleImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        label = int(self.labels[idx])

        if self.transform is not None:
            img = self.transform(img)

        return img, label

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [8]:
full_dataset_train = SimpleImageDataset(image_paths, labels, transform=train_transform)
full_dataset_val = SimpleImageDataset(image_paths, labels, transform=val_transform)

In [9]:
def build_model(model_name, num_classes=2):
    if model_name == "resnet50":
        model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "densenet121":
        model = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    return model.to(device)

In [10]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for imgs, targets in loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="binary")

    return epoch_loss, epoch_acc, epoch_f1


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            targets = targets.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * imgs.size(0)

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="binary")

    return epoch_loss, epoch_acc, epoch_f1, np.array(all_labels), np.array(all_preds)

In [11]:
def run_fold(model_name, train_idx, val_idx, epochs=10, batch_size=8, lr=1e-4, patience=3):
    train_dataset = Subset(full_dataset_train, train_idx)
    val_dataset = Subset(full_dataset_val, val_idx)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    model = build_model(model_name, num_classes=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_f1 = -1.0
    wait = 0

    history = []

    for epoch in range(epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1
        })

        print(
            f"[{model_name}] Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered for {model_name}.")
                break

    model.load_state_dict(best_model_wts)

    val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader, criterion)

    return {
        "model_name": model_name,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "y_true": y_true,
        "y_pred": y_pred,
        "history": pd.DataFrame(history)
    }

In [12]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for model_name in ["resnet50", "densenet121"]:
    print("\n" + "=" * 60)
    print(f"Running 5-fold CV for: {model_name}")
    print("=" * 60)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels), start=1):
        print("\n" + "-" * 40)
        print(f"Model: {model_name} | Fold {fold}")
        print("-" * 40)

        fold_result = run_fold(
            model_name=model_name,
            train_idx=train_idx,
            val_idx=val_idx,
            epochs=10,
            batch_size=8,
            lr=1e-4,
            patience=3
        )

        fold_metrics.append({
            "model": model_name,
            "fold": fold,
            "accuracy": fold_result["val_acc"],
            "f1": fold_result["val_f1"]
        })

        print("Fold accuracy:", fold_result["val_acc"])
        print("Fold F1:", fold_result["val_f1"])
        print(classification_report(fold_result["y_true"], fold_result["y_pred"], digits=4))

    fold_metrics_df = pd.DataFrame(fold_metrics)
    results.append(fold_metrics_df)


Running 5-fold CV for: resnet50

----------------------------------------
Model: resnet50 | Fold 1
----------------------------------------
[resnet50] Epoch 1/10 | train_loss=0.7083, train_acc=0.5000, train_f1=0.4667 | val_loss=0.6902, val_acc=0.5000, val_f1=0.5000
[resnet50] Epoch 2/10 | train_loss=0.5912, train_acc=0.9062, train_f1=0.9032 | val_loss=0.6636, val_acc=0.7500, val_f1=0.8000
[resnet50] Epoch 3/10 | train_loss=0.5356, train_acc=0.9688, train_f1=0.9677 | val_loss=0.6491, val_acc=0.6250, val_f1=0.7273
[resnet50] Epoch 4/10 | train_loss=0.4839, train_acc=1.0000, train_f1=1.0000 | val_loss=0.6493, val_acc=0.6250, val_f1=0.7273
[resnet50] Epoch 5/10 | train_loss=0.3764, train_acc=1.0000, train_f1=1.0000 | val_loss=0.6568, val_acc=0.6250, val_f1=0.7273
Early stopping triggered for resnet50.
Fold accuracy: 0.75
Fold F1: 0.8
              precision    recall  f1-score   support

           0     1.0000    0.5000    0.6667         4
           1     0.6667    1.0000    0.8000     

/home/amyliu/Desktop/GR/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/amyliu/Desktop/GR/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/amyliu/Desktop/GR/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

[resnet50] Epoch 1/10 | train_loss=0.7130, train_acc=0.4375, train_f1=0.1818 | val_loss=0.6892, val_acc=0.6250, val_f1=0.4000
[resnet50] Epoch 2/10 | train_loss=0.5787, train_acc=0.8750, train_f1=0.8571 | val_loss=0.6812, val_acc=0.3750, val_f1=0.2857
[resnet50] Epoch 3/10 | train_loss=0.4956, train_acc=1.0000, train_f1=1.0000 | val_loss=0.6846, val_acc=0.6250, val_f1=0.7273
[resnet50] Epoch 4/10 | train_loss=0.4427, train_acc=0.9062, train_f1=0.9032 | val_loss=0.7089, val_acc=0.5000, val_f1=0.5000
[resnet50] Epoch 5/10 | train_loss=0.3320, train_acc=0.9688, train_f1=0.9677 | val_loss=0.7236, val_acc=0.3750, val_f1=0.4444
[resnet50] Epoch 6/10 | train_loss=0.2440, train_acc=1.0000, train_f1=1.0000 | val_loss=0.7337, val_acc=0.2500, val_f1=0.4000
Early stopping triggered for resnet50.
Fold accuracy: 0.625
Fold F1: 0.7272727272727273
              precision    recall  f1-score   support

           0     1.0000    0.2500    0.4000         4
           1     0.5714    1.0000    0.7273    

In [13]:
results_df = pd.concat(results, ignore_index=True)
print(results_df)

         model  fold  accuracy        f1
0     resnet50     1     0.750  0.800000
1     resnet50     2     0.875  0.888889
2     resnet50     3     0.750  0.750000
3     resnet50     4     0.500  0.666667
4     resnet50     5     0.625  0.727273
5  densenet121     1     0.625  0.666667
6  densenet121     2     0.750  0.750000
7  densenet121     3     0.625  0.571429
8  densenet121     4     0.875  0.888889
9  densenet121     5     0.875  0.888889


In [14]:
summary_df = results_df.groupby("model").agg(
    mean_accuracy=("accuracy", "mean"),
    std_accuracy=("accuracy", "std"),
    mean_f1=("f1", "mean"),
    std_f1=("f1", "std")
).reset_index()

print(summary_df)

         model  mean_accuracy  std_accuracy   mean_f1    std_f1
0  densenet121           0.75      0.125000  0.753175  0.139070
1     resnet50           0.70      0.142522  0.766566  0.083485


In [15]:
best_by_acc = summary_df.sort_values("mean_accuracy", ascending=False).iloc[0]
best_by_f1 = summary_df.sort_values("mean_f1", ascending=False).iloc[0]

print("Best by accuracy:")
print(best_by_acc)

print("\nBest by F1:")
print(best_by_f1)

Best by accuracy:
model            densenet121
mean_accuracy           0.75
std_accuracy           0.125
mean_f1             0.753175
std_f1               0.13907
Name: 0, dtype: object

Best by F1:
model            resnet50
mean_accuracy         0.7
std_accuracy     0.142522
mean_f1          0.766566
std_f1           0.083485
Name: 1, dtype: object


## large daraset test

In [16]:
import os
from pathlib import Path
from PIL import Image
from torchvision.datasets import CIFAR10

# ===== paths =====
RAW_ROOT = Path("./data")
SAVE_ROOT = Path("./two_class_cifar_full")

SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# ===== CIFAR-10 label mapping =====
# airplane=0, automobile=1, bird=2, cat=3, deer=4,
# dog=5, frog=6, horse=7, ship=8, truck=9
target_classes = {
    3: "cat",
    5: "dog"
}

# ===== load datasets =====
train_dataset = CIFAR10(root=RAW_ROOT, train=True, download=True)
test_dataset = CIFAR10(root=RAW_ROOT, train=False, download=True)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

/home/amyliu/Desktop/GR/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train size: 50000
Test size: 10000


In [17]:
for split in ["train", "test"]:
    for class_name in ["cat", "dog"]:
        (SAVE_ROOT / split / class_name).mkdir(parents=True, exist_ok=True)

In [18]:
train_counts = {"cat": 0, "dog": 0}

for idx, (img, label) in enumerate(train_dataset):
    if label in target_classes:
        class_name = target_classes[label]
        save_path = SAVE_ROOT / "train" / class_name / f"{class_name}_{train_counts[class_name]:05d}.png"
        img.save(save_path)
        train_counts[class_name] += 1

print("Train exported:", train_counts)

Train exported: {'cat': 5000, 'dog': 5000}


In [19]:
test_counts = {"cat": 0, "dog": 0}

for idx, (img, label) in enumerate(test_dataset):
    if label in target_classes:
        class_name = target_classes[label]
        save_path = SAVE_ROOT / "test" / class_name / f"{class_name}_{test_counts[class_name]:05d}.png"
        img.save(save_path)
        test_counts[class_name] += 1

print("Test exported:", test_counts)

Test exported: {'cat': 1000, 'dog': 1000}


In [20]:
print("Dataset created at:", SAVE_ROOT.resolve())
print("Train cat:", len(list((SAVE_ROOT / "train" / "cat").glob("*.png"))))
print("Train dog:", len(list((SAVE_ROOT / "train" / "dog").glob("*.png"))))
print("Test cat:", len(list((SAVE_ROOT / "test" / "cat").glob("*.png"))))
print("Test dog:", len(list((SAVE_ROOT / "test" / "dog").glob("*.png"))))

Dataset created at: /home/amyliu/Desktop/GR/two_class_cifar_full
Train cat: 5000
Train dog: 5000
Test cat: 1000
Test dog: 1000


In [21]:
DATA_ROOT = Path("./two_class_cifar_full")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print("Dataset root:", DATA_ROOT.resolve())

Using device: cuda
Dataset root: /home/amyliu/Desktop/GR/two_class_cifar_full


In [22]:
class BinaryImageFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        class_to_idx = {"cat": 0, "dog": 1}

        for class_name, class_idx in class_to_idx.items():
            class_dir = self.root_dir / class_name
            for img_path in sorted(class_dir.glob("*.png")):
                self.samples.append((str(img_path), class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, label

In [23]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [24]:
train_dataset = BinaryImageFolderDataset(DATA_ROOT / "train", transform=train_transform)
test_dataset = BinaryImageFolderDataset(DATA_ROOT / "test", transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 10000
Test size: 2000


In [25]:
def build_model(model_name, num_classes=2, freeze_backbone=True):
    if model_name == "resnet50":
        model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "densenet121":
        model = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    else:
        raise ValueError("Unsupported model")

    return model.to(device)

In [26]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, epoch_f1


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, epoch_f1, np.array(all_labels), np.array(all_preds)

In [27]:
def train_model(model_name, epochs=5, lr=1e-3, freeze_backbone=True):
    model = build_model(model_name, freeze_backbone=freeze_backbone)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    best_model_wts = copy.deepcopy(model.state_dict())
    best_test_f1 = -1.0

    history = []

    for epoch in range(epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        test_loss, test_acc, test_f1, _, _ = evaluate(model, test_loader, criterion)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "test_f1": test_f1
        })

        print(
            f"[{model_name}] Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, train_f1={train_f1:.4f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}"
        )

        if test_f1 > best_test_f1:
            best_test_f1 = test_f1
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)

    test_loss, test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader, criterion)

    print("\nFinal test result:", model_name)
    print("Accuracy:", test_acc)
    print("F1:", test_f1)
    print(classification_report(y_true, y_pred, digits=4))

    return model, history, test_acc, test_f1

In [32]:
resnet_model, resnet_history, resnet_acc, resnet_f1 = train_model(
    model_name="resnet50",
    epochs=5,
    lr=1e-3,
    freeze_backbone=True
)

[resnet50] Epoch 1/5 | train_loss=0.4578, train_acc=0.7895, train_f1=0.7866 | test_loss=0.3993, test_acc=0.8175, test_f1=0.8064
[resnet50] Epoch 2/5 | train_loss=0.3762, train_acc=0.8331, train_f1=0.8303 | test_loss=0.3765, test_acc=0.8280, test_f1=0.8266
[resnet50] Epoch 3/5 | train_loss=0.3556, train_acc=0.8446, train_f1=0.8424 | test_loss=0.3604, test_acc=0.8390, test_f1=0.8387
[resnet50] Epoch 4/5 | train_loss=0.3456, train_acc=0.8482, train_f1=0.8456 | test_loss=0.3642, test_acc=0.8380, test_f1=0.8404
[resnet50] Epoch 5/5 | train_loss=0.3340, train_acc=0.8533, train_f1=0.8507 | test_loss=0.3616, test_acc=0.8280, test_f1=0.8289

Final test result: resnet50
Accuracy: 0.838
F1: 0.8403940886699507
              precision    recall  f1-score   support

           0     0.8485    0.8230    0.8355      1000
           1     0.8282    0.8530    0.8404      1000

    accuracy                         0.8380      2000
   macro avg     0.8383    0.8380    0.8380      2000
weighted avg     0.8

In [29]:
densenet_model, densenet_history, densenet_acc, densenet_f1 = train_model(
    model_name="densenet121",
    epochs=5,
    lr=1e-3,
    freeze_backbone=True
)

[densenet121] Epoch 1/5 | train_loss=0.4726, train_acc=0.7738, train_f1=0.7710 | test_loss=0.3868, test_acc=0.8210, test_f1=0.8102
[densenet121] Epoch 2/5 | train_loss=0.3788, train_acc=0.8300, train_f1=0.8267 | test_loss=0.3671, test_acc=0.8390, test_f1=0.8333
[densenet121] Epoch 3/5 | train_loss=0.3577, train_acc=0.8401, train_f1=0.8381 | test_loss=0.3580, test_acc=0.8465, test_f1=0.8404
[densenet121] Epoch 4/5 | train_loss=0.3563, train_acc=0.8378, train_f1=0.8352 | test_loss=0.3709, test_acc=0.8390, test_f1=0.8244
[densenet121] Epoch 5/5 | train_loss=0.3451, train_acc=0.8437, train_f1=0.8412 | test_loss=0.3540, test_acc=0.8485, test_f1=0.8436

Final test result: densenet121
Accuracy: 0.8485
F1: 0.8435725348477027
              precision    recall  f1-score   support

           0     0.8278    0.8800    0.8531      1000
           1     0.8719    0.8170    0.8436      1000

    accuracy                         0.8485      2000
   macro avg     0.8499    0.8485    0.8483      2000
w

In [30]:
print("ResNet50 -> Accuracy:", resnet_acc, "F1:", resnet_f1)
print("DenseNet121 -> Accuracy:", densenet_acc, "F1:", densenet_f1)

ResNet50 -> Accuracy: 0.836 F1: 0.8340080971659919
DenseNet121 -> Accuracy: 0.8485 F1: 0.8435725348477027


In [39]:
import copy
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [42]:
class BinaryDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        label_map = {"cat": 0, "dog": 1}

        for label, idx in label_map.items():
            folder = self.root_dir / label
            for img_path in sorted(folder.glob("*.png")):
                self.samples.append((str(img_path), idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [45]:
train_dataset = BinaryDataset(DATA_ROOT / "train", transform=train_transform)
test_dataset = BinaryDataset(DATA_ROOT / "test", transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 10000
Test size: 2000


In [44]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [37]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [43]:
train_dataset = BinaryDataset(DATA_ROOT / "train", transform=train_transform)
test_dataset = BinaryDataset(DATA_ROOT / "test", transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 10000
Test size: 2000


In [33]:
def build_efficientnet(num_classes=2, freeze_backbone=True):
    weights = EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model.to(device)

In [34]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return acc, f1


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return acc, f1, all_labels, all_preds

In [35]:
def train_efficientnet(epochs=5, lr=1e-3):
    model = build_efficientnet(freeze_backbone=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    best_f1 = -1
    best_model = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        test_acc, test_f1, _, _ = evaluate(model, test_loader)

        print(
            f"Epoch {epoch+1}: "
            f"Train Acc={train_acc:.4f}, F1={train_f1:.4f} | "
            f"Test Acc={test_acc:.4f}, F1={test_f1:.4f}"
        )

        if test_f1 > best_f1:
            best_f1 = test_f1
            best_model = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model)

    test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader)

    print("\nFinal Result (EfficientNet-B0)")
    print("Accuracy:", test_acc)
    print("F1:", test_f1)
    print(classification_report(y_true, y_pred, digits=4))

    return model, test_acc, test_f1

In [46]:
eff_model, eff_acc, eff_f1 = train_efficientnet(epochs=5, lr=1e-3)

Epoch 1: Train Acc=0.7701, F1=0.7666 | Test Acc=0.8215, F1=0.8163
Epoch 2: Train Acc=0.8089, F1=0.8055 | Test Acc=0.8235, F1=0.8189
Epoch 3: Train Acc=0.8139, F1=0.8105 | Test Acc=0.8335, F1=0.8317
Epoch 4: Train Acc=0.8164, F1=0.8135 | Test Acc=0.8245, F1=0.8315
Epoch 5: Train Acc=0.8246, F1=0.8222 | Test Acc=0.8375, F1=0.8283

Final Result (EfficientNet-B0)
Accuracy: 0.8335
F1: 0.831733198585144
              precision    recall  f1-score   support

           0     0.8266    0.8440    0.8352      1000
           1     0.8407    0.8230    0.8317      1000

    accuracy                         0.8335      2000
   macro avg     0.8336    0.8335    0.8335      2000
weighted avg     0.8336    0.8335    0.8335      2000

